# Hidden State Analysis for Transformer Models

This notebook analyzes hidden states across different layers of transformer models to understand how representations evolve and differ across tasks and prompt types.

## Purpose
- Analyze hidden state patterns across different model layers
- Compare representations across different prompt types
- Visualize task-specific clustering in hidden space
- Generate t-SNE plots for layer-wise analysis

## 1. Setup and Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.model_utils import load_model_and_tokenizer, collect_hidden_states
from src.visualization import plot_tsne_layers, plot_selected_layers

## 2. Configuration

In [ ]:
# Model and data paths
MODEL_NAME = "Llama-2-7b-hf"
MODELS_ROOT_PATH = "/mnt/public/model/huggingface/"
DATA_PATH = "../data/processed/prompts.parquet"

# Analysis parameters
PROMPT_TYPES = ["knowledge", "prompt_zero_shot", "prompt_cot", "prompt_icl", "corpus"]
TASK_FILTER = [
    'c4', 'wikitext2',           # Language modeling tasks
    'mbpp', 'humaneval',         # Code generation tasks
    'gsm8k', 'mathqa',          # Math reasoning tasks
    'arc_easy', 'arc_challenge', 'openbookqa',  # Knowledge QA tasks
    'winogrande', 'piqa', 'hellaswag',         # Common sense tasks
    'boolq', 'race'             # Reading comprehension tasks
]
SAMPLE_PER_TASK = 100

## 3. Helper Functions

In [ ]:
def build_prompt_hidden_dict(df: pd.DataFrame, model, tokenizer) -> list:
    """Build a dictionary mapping prompts to their hidden states.
    
    Args:
        df: DataFrame containing prompts and task information
        model: The transformer model
        tokenizer: The tokenizer for the model
        
    Returns:
        List of dictionaries containing prompt variants and their hidden states
    """
    results = []
    df = df.copy()
    df["id"] = [f"{row['task_type']}_{i}" for i, row in df.iterrows()]

    grouped = df.groupby("task_type")

    for task_type, group_df in grouped:
        print(f"\nTask: {task_type} | Samples: {len(group_df)}")
        for i, row in tqdm(group_df.iterrows(), total=len(group_df), desc=f"{task_type}"):
            prompt_variants = {}
            hidden_variants = {}

            for prompt_type in PROMPT_TYPES:
                prompt_text = row[prompt_type]
                prompt_variants[prompt_type] = prompt_text
                hidden_states, _ = collect_hidden_states(
                    [prompt_text], [row["task_type"]], model, tokenizer
                )
                hidden_variants[prompt_type] = hidden_states[0]  # single input

            results.append({
                "id": row["id"],
                "task_type": row["task_type"],
                "prompts": prompt_variants,
                "hidden_states": hidden_variants,
            })

    return results

## 4. Data Loading and Processing

In [ ]:
# Load and filter data
print("📦 Loading data...")
df = pd.read_parquet(DATA_PATH)
df = df[df["task_type"].isin(TASK_FILTER)]

# Filter to training split and sample
print("🎯 Filtering to training split only...")
train_df = df[df["split"] == "train"]

print("🎯 Sampling from each task...")
sampled = train_df.groupby("task_type").apply(
    lambda x: x.sample(n=min(len(x), SAMPLE_PER_TASK), random_state=42)
).reset_index(drop=True)

## 5. Model Loading and Hidden State Collection

In [ ]:
# Load model and collect hidden states
print("📥 Loading model...")
model_path = os.path.join(MODELS_ROOT_PATH, MODEL_NAME)
model, tokenizer = load_model_and_tokenizer(model_path)

print("🧠 Building prompt/hidden state pairs...")
result = build_prompt_hidden_dict(sampled, model, tokenizer)

## 6. Save/Load Results (Optional)

In [ ]:
import pickle

# Save results
cache_path = "../tmp/prompt_hidden.pkl"
with open(cache_path, "wb") as f:
    pickle.dump(result, f)
print(f"💾 Saved hidden states to {cache_path}")

# Load results if needed
with open(cache_path, "rb") as f:
    result = pickle.load(f)
print("✅ Loaded hidden states from cache")

## 7. Data Preparation for Visualization

In [ ]:
def flatten_results_by_prompt_type(results, prompt_types):
    """Flatten results for visualization.
    
    Args:
        results: List of result dictionaries
        prompt_types: List of prompt types to process
        
    Returns:
        Tuple of (hidden_states_list, task_types, prompt_tags)
    """
    hidden_states_list = []
    task_types = []
    prompt_tags = []

    for entry in results:
        task_type = entry["task_type"]
        for prompt_type in prompt_types:
            hidden = entry["hidden_states"].get(prompt_type)
            if hidden is not None:
                hidden_states_list.append(hidden)
                task_types.append(task_type)
                prompt_tags.append(prompt_type)

    return hidden_states_list, task_types, prompt_tags

# Prepare data for visualization
hidden_states_list, task_types, prompt_tags = flatten_results_by_prompt_type(result, PROMPT_TYPES)

## 8. Visualization

In [ ]:
# Generate t-SNE plots for each prompt type
for prompt_type in PROMPT_TYPES:
    filtered_states = [h for h, tag in zip(hidden_states_list, prompt_tags) if tag == prompt_type]
    filtered_tasks = [t for t, tag in zip(task_types, prompt_tags) if tag == prompt_type]
    
    print(f"🎨 Drawing t-SNE for {prompt_type} ...")
    plot_tsne_layers(filtered_states, filtered_tasks, perplexity=50)
    
    print(f"🎨 Drawing selected layer analysis for {prompt_type} ...")
    plot_selected_layers(filtered_states, filtered_tasks, perplexity=30)